# Library Management System Using Python

**Student name:** Abuhurairah Rahman Pathan  
**Platform:** Google Colab (Python 3)

## Project description
This project is a complete, menu-driven Library Management System that runs entirely in the Colab output console. A librarian can manage books, register members, issue and return books, search the catalogue and generate reports. All data is held in Python dictionaries, so no external libraries or databases are used.

## Objectives
- Store books, members and transactions in dictionaries and keep them updated correctly.
- Build each module as its own function, connected through a main menu driven by a `while` loop.
- Validate every input (unique IDs, 10-digit phone numbers, valid emails, positive copy counts).
- Use `try-except` so that typing text instead of a number never crashes the program.
- Record issue and return dates with the `datetime` module.
- Calculate every report dynamically from the live data.

## How to run
Select **Runtime → Run all**. The final cell starts the program. To reset the library to its original sample data, run the cells again from the top.

## Step 2: Initialise the library database
Three dictionaries hold all the data. Each **key** is a unique ID and each **value** is a nested dictionary of details.
- `books` – title, author, category, total copies and available copies.
- `members` – name, phone and email.
- `transactions` – starts empty; each issue creates a new record with status `Issued`, which later changes to `Returned`.

`issued copies = total − available`, so the program never needs to store the issued count separately.

In [ ]:
from datetime import datetime

# Books: key = Book ID
books = {
    101: {
        "title": "Python Programming",
        "author": "John Smith",
        "category": "Programming",
        "total": 5,
        "available": 5
    },
    102: {
        "title": "Data Science Basics",
        "author": "David Miller",
        "category": "Technology",
        "total": 3,
        "available": 3
    },
    103: {
        "title": "Database Management",
        "author": "Robert Brown",
        "category": "Database",
        "total": 4,
        "available": 4
    }
}

# Members: key = Member ID
members = {
    1: {
        "name": "Aarav Sharma",
        "phone": "9876543210",
        "email": "aarav@gmail.com"
    },
    2: {
        "name": "Meera Kapoor",
        "phone": "9876543211",
        "email": "meera@gmail.com"
    },
    3: {
        "name": "Rahul Verma",
        "phone": "9876543212",
        "email": "rahul@gmail.com"
    }
}

# Transactions: key = Transaction ID (starts empty)
# Each value: {"book_id", "member_id", "issue_date", "return_date", "status"}
transactions = {}

print("Library database loaded.")
print(f"Books: {len(books)} | Members: {len(members)} | Transactions: {len(transactions)}")

## Helper functions (input, validation and display)
These small functions are reused by every module, which keeps the code short and consistent.
- `get_int()` wraps `int(input())` in **try-except**. If the user types text, it prints an error and returns `None` instead of crashing. Every numeric input in the project (Book ID, Member ID, Transaction ID, copies, menu choices) goes through it.
- `get_text()` reads text and removes extra spaces with `.strip()`.
- `is_valid_email()` uses string methods to check for one `@`, a non-empty name part and a domain containing a dot.
- `show_book()`, `show_member()` and `show_transaction()` print records in a consistent format.

In [ ]:
def get_int(prompt):
    """Read a whole number safely. Returns None if the input is not a number."""
    try:
        return int(input(prompt).strip())
    except ValueError:
        print("Invalid input! Please enter a numeric value.")
        return None


def get_text(prompt):
    """Read text and remove leading/trailing spaces."""
    return input(prompt).strip()


def is_valid_email(email):
    """Check that an email contains one @ and a valid-looking domain, e.g. name@gmail.com."""
    if " " in email or email.count("@") != 1:
        return False
    local_part, domain = email.split("@")
    if local_part == "" or "." not in domain:
        return False
    if domain.startswith(".") or domain.endswith(".") or ".." in domain:
        return False
    return True


def show_book(book_id):
    book = books[book_id]
    print(f"Book ID: {book_id}")
    print(f"Title: {book['title']}")
    print(f"Author: {book['author']}")
    print(f"Category: {book['category']}")
    print(f"Total Copies: {book['total']}")
    print(f"Available Copies: {book['available']}")
    print("-" * 32)


def show_member(member_id):
    member = members[member_id]
    print(f"Member ID: {member_id}")
    print(f"Name: {member['name']}")
    print(f"Phone: {member['phone']}")
    print(f"Email: {member['email']}")
    print("-" * 32)


def show_transaction(txn_id):
    txn = transactions[txn_id]
    # .get() protects against a book or member that no longer exists
    title = books.get(txn["book_id"], {}).get("title", "[Book deleted]")
    name = members.get(txn["member_id"], {}).get("name", "[Member not found]")
    print(f"Transaction ID: {txn_id}")
    print(f"Book: {title} (ID {txn['book_id']})")
    print(f"Member: {name} (ID {txn['member_id']})")
    print(f"Issue Date: {txn['issue_date']}")
    print(f"Return Date: {txn['return_date'] if txn['return_date'] else 'Not returned yet'}")
    print(f"Status: {txn['status']}")
    print("-" * 32)

## Step 3: Main menu
`display_menu()` only prints the menu. The `while` loop that repeats it until the user chooses **Exit** is inside `main()` at the end of the notebook.

In [ ]:
def display_menu():
    print("\n========================================")
    print("       LIBRARY MANAGEMENT SYSTEM")
    print("========================================")
    print("1. Book Management")
    print("2. Member Management")
    print("3. Issue a Book")
    print("4. Return a Book")
    print("5. Search Books")
    print("6. Library Reports")
    print("7. Exit")
    print("========================================")

## Step 4: Book Management module
`book_management()` shows its own sub-menu in a `while` loop and calls one function per task.
- **Add** – Book ID must be a unique positive integer and copies must be greater than zero. Available copies start equal to total copies.
- **View** – a `for` loop goes through the `books` dictionary.
- **Update** – change title, author, category or total copies. Total copies can never go below the number currently issued, and available copies are recalculated as `new total − issued`.
- **Delete** – blocked if any copy is currently issued; otherwise the book is removed with `.pop()` after confirmation.

In [ ]:
def add_book():
    print("\n--- Add New Book ---")
    book_id = get_int("Enter Book ID: ")
    if book_id is None:
        return
    if book_id <= 0:
        print("Book ID must be a positive number.")
        return
    if book_id in books:
        print(f"Book ID {book_id} already exists. Please use a unique Book ID.")
        return

    title = get_text("Enter Book Title: ")
    author = get_text("Enter Author Name: ")
    category = get_text("Enter Category: ")
    if title == "" or author == "" or category == "":
        print("Title, author and category cannot be empty. Book not added.")
        return

    copies = get_int("Enter Number of Copies: ")
    if copies is None:
        return
    if copies <= 0:
        print("Number of copies must be greater than zero. Book not added.")
        return

    books[book_id] = {
        "title": title,
        "author": author,
        "category": category,
        "total": copies,
        "available": copies      # all copies are available at the start
    }
    print(f"Book '{title}' added successfully with ID {book_id}.")


def view_books():
    print("\n--- All Books ---")
    if not books:
        print("No books in the library.")
        return
    for book_id in books:
        show_book(book_id)


def update_book():
    print("\n--- Update Book Details ---")
    book_id = get_int("Enter Book ID: ")
    if book_id is None:
        return
    if book_id not in books:
        print(f"Book ID {book_id} does not exist.")
        return

    book = books[book_id]
    show_book(book_id)
    print("What would you like to update?")
    print("1. Title")
    print("2. Author")
    print("3. Category")
    print("4. Total Copies")
    print("5. Cancel")
    option = get_int("Enter your option: ")

    if option in (1, 2, 3):
        field = {1: "title", 2: "author", 3: "category"}[option]
        new_value = get_text(f"Enter new {field}: ")
        if new_value == "":
            print(f"{field.capitalize()} cannot be empty. No changes made.")
        else:
            old_value = book[field]
            book[field] = new_value
            print(f"{field.capitalize()} updated from '{old_value}' to '{new_value}'.")
    elif option == 4:
        issued = book["total"] - book["available"]
        new_total = get_int(f"Enter new total copies (currently issued: {issued}): ")
        if new_total is None:
            return
        if new_total <= 0:
            print("Total copies must be greater than zero.")
        elif new_total < issued:
            print(f"Not allowed! {issued} copies are currently issued, "
                  f"so total copies cannot be less than {issued}.")
        else:
            book["total"] = new_total
            book["available"] = new_total - issued
            print(f"Total copies updated to {new_total}. Available copies: {book['available']}.")
    elif option == 5:
        print("Update cancelled.")
    elif option is not None:
        print("Invalid option! Please select between 1 and 5.")


def delete_book():
    print("\n--- Delete Book ---")
    book_id = get_int("Enter Book ID: ")
    if book_id is None:
        return
    if book_id not in books:
        print(f"Book ID {book_id} does not exist.")
        return

    book = books[book_id]
    issued = book["total"] - book["available"]
    if issued > 0:
        print(f"Cannot delete '{book['title']}'. {issued} copy/copies are currently issued.")
        return

    confirm = get_text(f"Are you sure you want to delete '{book['title']}'? (y/n): ").lower()
    if confirm == "y":
        removed = books.pop(book_id)
        print(f"Book '{removed['title']}' (ID {book_id}) deleted successfully.")
    else:
        print("Deletion cancelled.")


def book_management():
    while True:
        print("\n-------- BOOK MANAGEMENT --------")
        print("1. Add New Book")
        print("2. View All Books")
        print("3. Update Book Details")
        print("4. Delete Book")
        print("5. Back to Main Menu")
        choice = get_int("Enter your choice: ")

        if choice == 1:
            add_book()
        elif choice == 2:
            view_books()
        elif choice == 3:
            update_book()
        elif choice == 4:
            delete_book()
        elif choice == 5:
            break
        elif choice is not None:
            print("Invalid choice! Please select between 1 and 5.")

## Step 5: Member Management module
- **Register** – Member ID must be unique, the name cannot be empty, the phone must be exactly 10 digits (`len()` and `.isdigit()`), and the email must pass `is_valid_email()`. Emails are stored in lower case.
- **View** – a `for` loop prints every member.
- **Search** – looks up the Member ID as a dictionary key and also shows how many books that member currently holds.

In [ ]:
def register_member():
    print("\n--- Register New Member ---")
    member_id = get_int("Enter Member ID: ")
    if member_id is None:
        return
    if member_id <= 0:
        print("Member ID must be a positive number.")
        return
    if member_id in members:
        print(f"Member ID {member_id} already exists. Please use a unique Member ID.")
        return

    name = get_text("Enter Full Name: ")
    if name == "":
        print("Name cannot be empty. Member not registered.")
        return

    phone = get_text("Enter Phone Number (10 digits): ")
    if len(phone) != 10 or not phone.isdigit():
        print("Phone number must contain exactly 10 digits. Member not registered.")
        return

    email = get_text("Enter Email Address: ").lower()
    if not is_valid_email(email):
        print("Invalid email address (example: name@gmail.com). Member not registered.")
        return

    members[member_id] = {"name": name, "phone": phone, "email": email}
    print(f"Member '{name}' registered successfully with ID {member_id}.")


def view_members():
    print("\n--- All Members ---")
    if not members:
        print("No members registered.")
        return
    for member_id in members:
        show_member(member_id)


def search_member():
    print("\n--- Search Member ---")
    member_id = get_int("Enter Member ID: ")
    if member_id is None:
        return
    if member_id not in members:
        print(f"Member ID {member_id} is not registered.")
        return

    print("Member found:")
    show_member(member_id)
    holding = 0
    for txn in transactions.values():
        if txn["member_id"] == member_id and txn["status"] == "Issued":
            holding += 1
    print(f"Books currently issued to this member: {holding}")


def member_management():
    while True:
        print("\n-------- MEMBER MANAGEMENT --------")
        print("1. Register New Member")
        print("2. View All Members")
        print("3. Search Member")
        print("4. Back to Main Menu")
        choice = get_int("Enter your choice: ")

        if choice == 1:
            register_member()
        elif choice == 2:
            view_members()
        elif choice == 3:
            search_member()
        elif choice == 4:
            break
        elif choice is not None:
            print("Invalid choice! Please select between 1 and 4.")

## Step 6: Issue a Book
Checks run in order, and the function stops at the first failure: the book exists → it has an available copy → the member is registered. Only then is a new transaction created, today's date recorded and the available count reduced by one.

`generate_transaction_id()` starts at **1001** and then uses the highest existing ID + 1, so every issue gets its own unique transaction.

In [ ]:
def generate_transaction_id():
    if not transactions:
        return 1001
    return max(transactions.keys()) + 1


def issue_book():
    print("\n--- Issue a Book ---")
    book_id = get_int("Enter Book ID: ")
    if book_id is None:
        return
    if book_id not in books:
        print(f"Invalid Book ID! Book {book_id} does not exist.")
        return

    book = books[book_id]
    if book["available"] <= 0:
        print(f"Sorry, '{book['title']}' is not available. All {book['total']} copies are issued.")
        return

    member_id = get_int("Enter Member ID: ")
    if member_id is None:
        return
    if member_id not in members:
        print(f"Member ID {member_id} is not registered. Book cannot be issued.")
        return

    txn_id = generate_transaction_id()
    transactions[txn_id] = {
        "book_id": book_id,
        "member_id": member_id,
        "issue_date": datetime.now().strftime("%d-%m-%Y"),
        "return_date": None,
        "status": "Issued"
    }
    book["available"] -= 1

    print("\nBook Issued Successfully!")
    print(f"Transaction ID: {txn_id}")
    print(f"Book: {book['title']}")
    print(f"Member: {members[member_id]['name']}")
    print(f"Issue Date: {transactions[txn_id]['issue_date']}")
    print(f"Status: Issued")

## Step 7: Return a Book
The transaction must exist and its status must be `Issued`. If it is already `Returned`, the second return is rejected. On success the status changes, the return date is recorded and the available count goes up by one; the `if` check ensures available copies can never exceed total copies.

In [ ]:
def return_book():
    print("\n--- Return a Book ---")
    txn_id = get_int("Enter Transaction ID: ")
    if txn_id is None:
        return
    if txn_id not in transactions:
        print(f"Transaction ID {txn_id} does not exist.")
        return

    txn = transactions[txn_id]
    if txn["status"] == "Returned":
        print(f"This book was already returned on {txn['return_date']}. "
              f"A transaction cannot be returned twice.")
        return

    txn["status"] = "Returned"
    txn["return_date"] = datetime.now().strftime("%d-%m-%Y")

    book = books.get(txn["book_id"])
    if book is not None and book["available"] < book["total"]:
        book["available"] += 1

    title = book["title"] if book else "[Book deleted]"
    print("\nBook Returned Successfully!")
    print(f"Book: {title}")
    print(f"Member: {members.get(txn['member_id'], {}).get('name', '[Member not found]')}")
    print(f"Return Date: {txn['return_date']}")
    print(f"Status: Returned")

## Step 8: Search Books
- **By ID** – direct dictionary key lookup (`book_id in books`).
- **By title, author or category** – the search text and the stored text are both converted with `.lower()`, and `in` checks for a partial match. So searching `python` finds *Python Programming*.
- A list comprehension collects every matching Book ID; if the list is empty, a "no record found" message is shown.

In [ ]:
def search_books():
    while True:
        print("\n-------- SEARCH BOOKS --------")
        print("1. Search by Book ID")
        print("2. Search by Book Title")
        print("3. Search by Author")
        print("4. Search by Category")
        print("5. Back to Main Menu")
        choice = get_int("Enter your choice: ")

        if choice == 1:
            book_id = get_int("Enter Book ID: ")
            if book_id is None:
                continue
            if book_id in books:
                print("\nBook found:")
                show_book(book_id)
            else:
                print(f"No record found for Book ID {book_id}.")

        elif choice in (2, 3, 4):
            field = {2: "title", 3: "author", 4: "category"}[choice]
            keyword = get_text(f"Enter {field} to search: ").lower()
            if keyword == "":
                print("Search text cannot be empty.")
                continue
            matches = [book_id for book_id in books if keyword in books[book_id][field].lower()]
            if matches:
                print(f"\n{len(matches)} matching book(s) found:")
                for book_id in matches:
                    show_book(book_id)
            else:
                print(f"No record found for {field} '{keyword}'.")

        elif choice == 5:
            break
        elif choice is not None:
            print("Invalid choice! Please select between 1 and 5.")

## Step 9: Library Reports
Every figure is **calculated live** from the dictionaries with loops and conditions – nothing is hard-coded.
1. Unique titles = `len(books)`
2. Total copies = sum of every book's `total`
3. Available copies = sum of every book's `available`
4. Currently issued = transactions whose status is `Issued`
5. Registered members = `len(members)`
6. Full transaction history with book title, member name, dates and status

In [ ]:
def library_reports():
    while True:
        print("\n-------- LIBRARY REPORTS --------")
        print("1. Total Books in Library")
        print("2. Total Book Copies")
        print("3. Available Book Copies")
        print("4. Currently Issued Books")
        print("5. Total Registered Members")
        print("6. Transaction History")
        print("7. Back to Main Menu")
        choice = get_int("Enter your choice: ")

        if choice == 1:
            print(f"\nTotal unique book titles in library: {len(books)}")

        elif choice == 2:
            total_copies = 0
            for book in books.values():
                total_copies += book["total"]
            print(f"\nTotal book copies (all titles): {total_copies}")

        elif choice == 3:
            available_copies = 0
            for book in books.values():
                available_copies += book["available"]
            print(f"\nAvailable book copies: {available_copies}")

        elif choice == 4:
            print("\n--- Currently Issued Books ---")
            issued_count = 0
            for txn_id, txn in transactions.items():
                if txn["status"] == "Issued":
                    show_transaction(txn_id)
                    issued_count += 1
            if issued_count == 0:
                print("No books are currently issued.")
            else:
                print(f"Total currently issued: {issued_count}")

        elif choice == 5:
            print(f"\nTotal registered members: {len(members)}")

        elif choice == 6:
            print("\n--- Transaction History ---")
            if not transactions:
                print("No transactions recorded yet.")
            else:
                for txn_id in transactions:
                    show_transaction(txn_id)
                print(f"Total transactions: {len(transactions)}")

        elif choice == 7:
            break
        elif choice is not None:
            print("Invalid choice! Please select between 1 and 7.")

## Final integration: main menu driver
`main()` loops forever with `while True`, shows the menu, reads the choice inside **try-except** and calls the matching module. After each module finishes, control returns here. Option 7 uses `break` to leave the loop and end the program; any other number shows an error, and text input is caught by `except ValueError`.

**Run this cell to start the program.**

In [ ]:
def main():
    while True:
        display_menu()
        try:
            choice = int(input("Enter your choice: "))
            if choice == 1:
                book_management()
            elif choice == 2:
                member_management()
            elif choice == 3:
                issue_book()
            elif choice == 4:
                return_book()
            elif choice == 5:
                search_books()
            elif choice == 6:
                library_reports()
            elif choice == 7:
                print("Thank you for using Library Management System!")
                break
            else:
                print("Invalid choice! Please select between 1 and 7.")
        except ValueError:
            print("Please enter a valid numeric choice.")


main()

## Testing results
| # | Test case | How to test | Expected result |
|---|---|---|---|
| 1 | Add a new book | 1 → 1 → ID `104`, details, copies `2` | Book added |
| 2 | Add an existing Book ID | 1 → 1 → ID `101` | Duplicate ID rejected |
| 3 | View all books | 1 → 2 | All books displayed |
| 4 | Update a book | 1 → 3 → `104` → 1 → new title | Title changes |
| 5 | Delete a book with no issued copies | 1 → 4 → `104` → `y` | Book deleted |
| 6 | Register a new member | 2 → 1 → ID `4`, valid details | Member registered |
| 7 | Register an existing Member ID | 2 → 1 → ID `1` | Duplicate ID rejected |
| 8 | Issue an available book | 3 → `101` → `1` | Transaction 1001 created |
| 9 | Issue an unavailable book | Issue `102` three times, then a fourth | Fourth issue rejected |
| 10 | Issue to an invalid member | 3 → `101` → `99` | Operation rejected |
| 11 | Return an issued book | 4 → `1001` | Availability increases |
| 12 | Return the same book twice | 4 → `1001` again | Second return rejected |
| 13 | Search for an existing title | 5 → 2 → `python` | *Python Programming* displayed |
| 14 | Search for a nonexistent book | 5 → 2 → `chemistry` | "No record found" message |
| 15 | Select an invalid menu option | Main menu → `9` | Error message appears |
| 16 | Enter text instead of a number | Main menu → `abc` | Error message, program keeps running |

**Extra rules also tested:** total copies cannot be set below issued copies; a book with issued copies cannot be deleted; phone numbers must be exactly 10 digits; invalid emails are rejected.